In [ ]:
import pickle
from pathlib import Path

import getdist as gd
import getdist.plots as gdplots
import numpy as np
from ris.estimate import compute_harmonic_mean, ln_evidence_from_ln_inverse

from expconfig import ExpConfig

In [ ]:
OUTPUTS_PATH = Path().resolve().parent / "outputs"


def get_results_dir(n: int = 0) -> Path:
    """Get the most recent results directory.

    Parameters
    ----------
    n : int, optional
        Index of the results directory to retrieve.
        0 for the most recent, 1 for the second most recent, etc.
        Default is 0.

    Returns
    -------
    Path
        Path to the results directory.
    """
    output_dirs = sorted(
        OUTPUTS_PATH.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True
    )
    return output_dirs[n]

In [ ]:
RESULTS_DIR = get_results_dir(0)

cfg = ExpConfig.load(RESULTS_DIR / "config.yaml")

In [ ]:
ln_prob = pickle.load(open(RESULTS_DIR / "lnprob.pkl", "rb"))
samples = pickle.load(open(RESULTS_DIR / "samples.pkl", "rb"))
ln_prob.shape, samples.shape

In [ ]:
print(ndim := samples.shape[-1])

In [ ]:
# very annoying reshaping - model training requires 2D, inference requires 3D
samples = samples.reshape((cfg.sampling.nwalkers, -1, ndim))
ln_prob = ln_prob.reshape((cfg.sampling.nwalkers, -1))

In [ ]:
import matplotlib.pyplot as plt

plt.plot(ln_prob.swapaxes(0, 1))

In [ ]:
flow_config = cfg.flow
train_config = cfg.training

train_samples = samples[:, ::2, :].reshape((-1, ndim))
test_samples = samples[:, 1::2]
train_ln_prob = ln_prob[:, ::2].reshape(-1)
test_ln_prob = ln_prob[:, 1::2]

In [ ]:
train_samples_gd = gd.MCSamples(samples=train_samples)
test_samples_gd = gd.MCSamples(samples=test_samples.reshape((-1, ndim)))
g = gdplots.getSubplotPlotter()
g.triangle_plot(
    [train_samples_gd, test_samples_gd],
    legend_labels=["Train", "Test"],
    filled=True,
)

In [ ]:
model_cls = flow_config.flow_model_config.model_cls()
flow_kwargs = flow_config.model_dump(exclude={"flow_model_config", "flow_type"})
flow_kwargs["temperature"] = (
    0.85  # This is fixed at 1 in FlowConfig for sddr calculations, so force override for harmonic
)
model_kwargs = flow_config.flow_model_config.model_dump()
model = model_cls(ndim_in=ndim, **model_kwargs, **flow_kwargs)
model.fit(X=train_samples, **train_config.model_dump())

In [ ]:
new_samples = model.sample(train_samples.shape[0])
new_samples_gd = gd.MCSamples(samples=new_samples)
g = gdplots.getSubplotPlotter()
g.triangle_plot(
    [train_samples_gd, new_samples_gd],
    legend_labels=["Train", "From Model"],
    filled=True,
)

In [ ]:
ln_evidence, ln_std_evidence = ln_evidence_from_ln_inverse(
    *compute_harmonic_mean(test_samples, test_ln_prob, model)[:-1]
)
ln_evidence_std = np.exp(
    ln_std_evidence - ln_evidence
)  # first order estimate of stddev of log of evidence
print(f"ln(Z) = {ln_evidence}±{ln_evidence_std}")